# 🦜🔗 Building LLM Applications with LangChain
### A hands-on, self-paced notebook using open-source models in Google Colab

Welcome! This notebook is a guided tour of **LangChain**, the most popular framework for building
applications powered by Large Language Models (LLMs). You will go from "calling a model" all the
way up to a working **Retrieval-Augmented Generation (RAG)** application — writing and running every
piece of code yourself.

The notebook is designed to be **self-explanatory**: every code cell is preceded by a short
explanation of *what* you're about to do and *why*, and followed (where useful) by a note on what
to look for in the output. You should be able to work through it top to bottom without needing
outside material.

## What you will learn
| # | Module | Key concepts |
|---|--------|---------------|
| 1 | Setup | Installing LangChain, authenticating with Hugging Face & Groq |
| 2 | Your First LLM Calls | Open-source models, the Hugging Face `pipeline` |
| 3 | Prompt Templates | `PromptTemplate`, `ChatPromptTemplate`, chat models |
| 4 | Few-Shot Prompting | Teaching a model by example |
| 5 | Chains (LCEL) | The `\|` operator, sequential chains, output parsers |
| 6 | Agents & Tools | Built-in tools (Wikipedia), custom tools, reasoning agents |
| 7 | Retrieval-Augmented Generation | Loading documents, splitting text, embeddings, vector stores |

## Prerequisites
- Basic Python (functions, dictionaries, f-strings)
- A free **[Hugging Face](https://huggingface.co/join)** account and access token
- A free **[Groq](https://console.groq.com/keys)** account and API key (Groq gives us fast,
  free-tier inference for larger open-source chat models — more on this in Module 3)
- Nothing else to install locally — everything runs in this Colab notebook

## A note on "open source" in this notebook
This notebook uses **open-source model weights** throughout (Meta's Llama 3.2, `nano-mistral`,
OpenAI's open-weight `gpt-oss`, `sentence-transformers` embeddings). Two different ways of
*running* those weights are shown:
1. **Locally, inside Colab**, via 🤗 Hugging Face's `pipeline` (Module 2) — great for understanding
   what's happening, but slow on Colab's free CPU/GPU for anything bigger than a few billion
   parameters.
2. **Hosted, via Groq's free inference API** (Module 3 onward) — same open-weight models, but
   served on Groq's specialized hardware so responses come back in a second instead of a minute.
   This is what most of the notebook uses once we get past the basics, so the demos stay fast.

## How to use this notebook
- Run the cells **in order** — later cells build on variables created earlier (e.g. `chat_model`,
  `retriever`).
- Cells marked **🔧 Configure** need a small edit from you (an API key, a file path) before they'll run.
- Cells marked **🧪 Try it yourself** are optional mini-exercises — great for reinforcing the concept
  you just saw.
- If Colab disconnects or you restart the runtime, re-run from the top of the relevant Module —
  each Module is written to be runnable as a unit as long as Module 1 (Setup) has been run first.

**Tip:** Go to `Runtime → Change runtime type` and select a **GPU** (e.g. T4) before starting —
it will noticeably speed up the local Hugging Face model in Module 2.

---

## Module 1 — Setup

Before we touch LangChain, we need to:
1. Install the LangChain packages this notebook uses.
2. Authenticate with Hugging Face (to download open-source model weights).
3. Later (Module 3), authenticate with Groq (to call hosted open-source models quickly).

We install three packages:
- **`langchain-huggingface`** — run open-source Hugging Face models directly on your Colab machine.
- **`langchain-groq`** — call open-source models hosted on Groq's fast inference servers.
- **`langchain-community`** — a grab-bag of community-maintained integrations (document loaders,
  tools, etc.) that we'll use later for RAG and agents.

In [ ]:
!pip install langchain-huggingface langchain-groq langchain-community


### Authenticate with Hugging Face

`HuggingFacePipeline` needs to download model weights from the Hugging Face Hub, and some models
(like Llama 3.2) are "gated" — you must accept their license on the model's Hugging Face page
*before* you can download them. Logging in here lets the notebook download any weights your
account has access to.

Run the cell below, then paste your [Hugging Face access token](https://huggingface.co/settings/tokens)
into the box that appears (a token with "Read" permission is enough).

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from huggingface_hub import login


In [ ]:
# 🔧 Configure: running this opens a login widget — paste your Hugging Face access token
login()


---
## Module 2 — Your First LLM Calls (Open-Source Models Locally)

`HuggingFacePipeline` wraps any Hugging Face `transformers` pipeline so it behaves like a LangChain
model. That means once we've created it, we call it the *same way* we'll call every other model in
this notebook: `.invoke(prompt)`.

Below, we load **Llama 3.2 (3B, Instruct-tuned)** — a genuinely capable open-source model, small
enough to run in Colab. `from_model_id(...)` downloads the weights the first time it runs (this can
take a couple of minutes), then keeps them cached for reuse.

- `model_id`: the model's identifier on the Hugging Face Hub
- `task`: what the pipeline should do — `"text-generation"` for a plain autoregressive LLM
- `pipeline_kwargs`: passed straight through to the underlying `transformers` pipeline;
  `max_new_tokens=100` caps how long the reply can be (longer replies = slower + more compute)

In [ ]:
llama_3_2 = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.2-3B-Instruct",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 100}
)


Now let's actually call it. `.invoke()` is the standard LangChain method for "run this model/chain on this input and give me the result" — you'll see it everywhere from here on.

In [ ]:
llama_3_2.invoke("What is Hugging Face?")


**What to expect:** a short paragraph answering the question. Since this is a *raw* text-completion
model call (no chat formatting yet), the model may continue rambling past a clean answer — we'll
fix that with prompt templates and chat-formatted models in Module 3.

#### Ultra-light LLM

Not every task needs a 3-billion-parameter model. `crumb/nano-mistral` is a tiny model, useful when
you want fast iteration or are extremely constrained on compute. It's not going to write you an
essay, but it demonstrates that the *exact same LangChain code* works regardless of model size —
that's the point of the abstraction.

In [ ]:
# Define the LLM from the Hugging Face model ID
nano_mistral = HuggingFacePipeline.from_model_id(
    model_id="crumb/nano-mistral",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 20}
)

prompt = "Hugging Face is"

# Invoke the model
response = nano_mistral.invoke(prompt)
print(response)


🧪 **Try it yourself:** Change `prompt` above to a different sentence starter and re-run the cell.
Notice how much less coherent `nano-mistral`'s completions are compared to Llama 3.2 — a small,
concrete illustration of why model size and training matter.

---
## Module 3 — Prompt Templates

So far we've sent raw strings to the model. In real applications, the *text* usually comes from a
user (or another part of your app) while the *instructions around it* stay fixed. **Prompt
templates** let you separate the two: you write a template with placeholders once, then fill in
the placeholders every time you call it.

`PromptTemplate` is the simplest kind — a plain string with `{variable}` placeholders.

In [ ]:
from langchain_core.prompts import PromptTemplate


Below, `prompt_template.invoke({...})` fills in the placeholder and returns a formatted prompt —
try printing `prompt` to see the filled-in string before it's sent to a model.

The `|` (pipe) operator is LangChain's way of **chaining** components together: `prompt_template | llama_3_2`
creates a mini-pipeline where the output of the template automatically becomes the input to the
model. This is the core of what's called **LCEL** (LangChain Expression Language) — we'll build
longer chains with it in Module 5.

In [ ]:
template = "Expain this concept simply and concisely: {concept}"
prompt_template = PromptTemplate.from_template(template=template)
prompt = prompt_template.invoke({"concept": "Prompting LLMs"})
print(prompt)

llm_chain = prompt_template | llama_3_2
concept = "Prompting LLMs"
print(llm_chain.invoke({"concept": concept}))


### Chat Models

Plain text completion models (like the ones above) just continue whatever text they're given.
**Chat models** instead work with a *list of turns* — `system` (instructions for the model),
`human` (the user), and `ai` (the model's own past replies) — which is how ChatGPT-style
conversational behavior is achieved.

From here on we'll switch to a **chat model hosted on Groq**. Groq serves open-weight models
(here, OpenAI's open-source `gpt-oss-120b`) on custom inference hardware, so responses come back
almost instantly — much better for interactive demos than running a 120B-parameter model locally
in Colab.

`google.colab.userdata` is Colab's built-in secret storage — it lets you store an API key once
(via the 🔑 key icon in the left sidebar) without ever typing it into a cell or committing it to
the notebook.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata
from langchain_groq import ChatGroq
import os


### 🔧 Configure: add your Groq API key

1. Get a free API key from [console.groq.com/keys](https://console.groq.com/keys).
2. In Colab, click the 🔑 **key icon** in the left sidebar → **Add new secret**.
3. Name it `GROQ_API_KEY`, paste your key as the value, and toggle "Notebook access" on.
4. Run the cell below.

In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
chat_model = ChatGroq(model="openai/gpt-oss-120b", api_key=os.environ["GROQ_API_KEY"])


`chat_model` will be reused for the rest of the notebook (chains, agents, RAG) — think of it as
"the LLM" that all our later components talk to.

`ChatPromptTemplate.from_messages([...])` builds a conversation template out of `(role, text)`
tuples. Notice the pattern below: we give the model **one worked example** (`human` → `ai`) before
asking our real question — this nudges the model toward the exact answer format we want (a
one-shot example is the simplest form of the "few-shot prompting" technique we'll formalize in
Module 4).

In [ ]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a calculator that responds with math."),
         ("human", "Answer this math question: What is two plus two?"),
          ("ai", "2+2=4"),
           ("human", "Answer this math question: {math}")
           ]
    )

llm_chain = template | chat_model
math = 'What is five times five?'
response = llm_chain.invoke({"math": math})
print(response.content)


**Note:** chat models return a message object, not a plain string — that's why we access `response.content` instead of printing `response` directly.

In [ ]:
# Create a chat prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a geography expert that returns the colors present in a country's flag."),
        ("human", "France"),
        ("ai", "blue, white, red"),
        ("human", "{country}")
    ]
)

# Chain the prompt template and model, and invoke the chain
llm_chain = prompt_template | chat_model

country = "Japan"
response = llm_chain.invoke({"country": country})
print(response.content)


🧪 **Try it yourself:** Change `country` to a few different countries. Because of the one-shot example, the model should reliably reply with just a comma-separated color list — no extra commentary.

---
## Module 4 — Few-Shot Prompting

The "one example" trick above works, but for trickier tasks you often want to show the model
**several** examples of input → output pairs. Rather than hand-writing them into a chat template
every time, `FewShotPromptTemplate` lets you keep a reusable list of examples and automatically
formats them into the prompt.

This is especially useful when:
- The desired output format is unusual or hard to describe in words
- You have a growing bank of examples you want to experiment with
- You want to swap in different examples without rewriting the whole prompt

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate


First, define a small list of example question/answer pairs — these are the "lessons" the model will learn the pattern from:

In [ ]:
# Create the examples list of dicts
examples = [
  {
    "question": "How many DataCamp courses has Jack completed?",
    "answer": "36"
  },
  {
    "question": "How much XP does Jack have on DataCamp?",
    "answer": "284,320XP"
  },
  {
    "question": "What technology does Jack learn about most on DataCamp?",
    "answer": "Python"
  }
]


Next, `example_prompt` defines *how each individual example is formatted*, and `FewShotPromptTemplate`
stitches all the formatted examples together, followed by a `suffix` where the real question goes.
Print `prompt` below to see the full assembled few-shot prompt before it's sent to the model.

In [ ]:
# Complete the prompt for formatting answers
example_prompt = PromptTemplate.from_template("Question: {question}\n{answer}")

# Create the few-shot prompt
prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {input}",
    input_variables=["input"],
)

prompt = prompt_template.invoke({"input": "What is Jack's favorite technology on DataCamp?"})

# Create and invoke the chain
llm_chain = prompt_template | chat_model
print(llm_chain.invoke({"input": "What is Jack's favorite technology on DataCamp?"}))


**What to expect:** a short, direct answer in the same terse style as the examples (e.g. `"Python"`), even though the model was never told explicitly to be brief — it inferred the style from the examples.

---
## Module 5 — Chains and Agents

So far every chain has been "prompt → model". Real applications are usually longer pipelines:
parse output, call a second model, look something up, make a decision. This module covers two ways
to build that: **LCEL chains** (you define the exact sequence of steps) and **agents** (the LLM
decides which steps to take, in what order, using tools you give it).

### Sequential Chains

`StrOutputParser` converts a chat model's message object into a plain string — handy when you want
to feed a model's output straight into *another* prompt template, which expects plain text, not a
message object.

In [ ]:
from langchain_core.output_parsers import StrOutputParser


Below we build a two-step travel-planning pipeline:
1. `destination_prompt` asks the model for activity ideas for a destination.
2. `activities_prompt` takes *those activities* and asks the model to turn them into a one-day itinerary.

The dictionary `{"activities": destination_prompt | chat_model | StrOutputParser()}` is LCEL syntax
for "run this first chain, and put its string output into the `activities` variable" — that
variable is then what `activities_prompt`'s `{activities}` placeholder gets filled with. Reading it
top to bottom: destination → activities (chain 1) → itinerary (chain 2).

In [ ]:
destination_prompt = PromptTemplate(input_variables=["destination"],
                                    template="I am planning a trip to {destination}. Can you suggest some activities to do there?")

activities_prompt = PromptTemplate(input_variables=["activities"],
                                   template="I only have one day, so can you create an itinerary from your top three activities: {activities}.")

seq_chain = ({"activities": destination_prompt | chat_model | StrOutputParser()}
| activities_prompt
| chat_model
| StrOutputParser())

print(seq_chain.invoke({"destination": "Rome"}))


**What to expect:** a one-day itinerary for Rome, even though we only ever gave the chain a
destination — the intermediate "suggest activities" step happened automatically inside the chain.

🧪 **Try it yourself:** Change `"Rome"` to a city of your choice and re-run.

### Introduction to LangChain Agents

Chains follow a **fixed** sequence of steps that *you* designed. **Agents** flip this around: you
give the LLM a set of **tools** (functions it can call) and a goal, and the LLM itself decides
which tool(s) to call, in which order, based on the model's reasoning about what the query needs.
This is the foundation of "agentic" LLM applications.

In [ ]:
!pip install mypy-extensions wikipedia


In [ ]:
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools


#### Using a Built-in Tool: Wikipedia

`load_tools(["wikipedia"])` gives us a ready-made tool that searches Wikipedia. Every LangChain
tool has a `name` and a `description` — the **description is what the LLM reads** to decide
whether and when to use the tool, so descriptive tool docstrings matter a lot in practice.

`create_agent(model, tools)` builds a ReAct-style agent: given a question, the model can choose to
call the Wikipedia tool (possibly more than once), read the results, and then compose a final
answer.

In [ ]:
# Define the tools
tools = load_tools(["wikipedia"])
print(tools[0].name)
print(tools[0].description)

# Define the agent
agent = create_agent(chat_model, tools)

# Invoke the agent
response = agent.invoke({"messages": [("human", "How many people live in New York City?")]})
print(response['messages'][-1].content)


**What to expect:** the printed tool name/description, followed by an answer citing a population
figure — pulled from a live Wikipedia lookup the agent performed on its own, not from the model's
training data.

### Using Custom Tools

You aren't limited to built-in tools — any Python function can become a tool with the `@tool`
decorator. The function's **docstring becomes its description**, so write it the way you'd explain
the function to a colleague; the LLM relies on it entirely to know when to call the tool.

In [ ]:
from langchain_core.tools import tool


In [ ]:
@tool
def financial_report(company_name: str, revenue: int, expenses: int) -> str:
  """Generate a financial report for a company that calculates net income."""
  net_income = revenue - expenses
  report = f"Financial Report for {company_name}:\n"
  report += f"Revenue: ${revenue}\n"
  report += f"Expenses: ${expenses}\n"
  report += f"Net Income: ${net_income}\n"
  return report


Before wiring it into an agent, let's inspect what LangChain automatically extracted from the function: its name, description, whether its output should be returned directly to the user, and its expected arguments.

In [ ]:
# Examining new tool
print(financial_report.name)
print(financial_report.description)
print(financial_report.return_direct)
print(financial_report.args)


In [ ]:
agent = create_agent(chat_model, [financial_report])
messages = agent.invoke({"messages": [("human", "TechStack generated made $10 million with $8 million of costs. Generate a financial report.")]})
print(messages)

print(messages['messages'][-1].content)


**What to expect:** the raw `messages` dict shows the full back-and-forth (the agent's tool call,
the tool's return value, and the final reply); the second `print` isolates just the human-readable
final report.

Custom tools can wrap *anything* — including a lookup over your own data. Here we give the agent
access to a small in-memory customer database and a tool to query it by name.

In [ ]:
import pandas as pd


In [ ]:
# Define the data
data = {
    "id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "name": [
        "Tech Innovators Inc.",
        "Green Solutions Ltd.",
        "Global Enterprises",
        "Peak Performance Co.",
        "Visionary Ventures",
        "NextGen Technologies",
        "Dynamic Dynamics LLC",
        "Infinity Services",
        "Eco-Friendly Products",
        "Future Insights"
    ],
    "subscription_type": [
        "Premium", "Standard", "Basic", "Premium", "Standard",
        "Basic", "Premium", "Standard", "Basic", "Premium"
    ],
    "engagement_score": [85, 60, 45, 90, 65, 40, 88, 55, 42, 92],
    "renewal_probability": [0.9, 0.6, 0.4, 0.95, 0.62, 0.35, 0.92, 0.58, 0.38, 0.96],
}

customers = pd.DataFrame(data)
customers.head()


In [ ]:
@tool
def retrieve_customer_info(name: str) -> str:
    """Retrieve customer information based on their name."""
    customer_info = customers[customers['name'] == name]
    return customer_info.to_string()

# Create a ReAct agent
agent = create_agent(chat_model, [retrieve_customer_info])

# Invoke the agent on the input
messages = agent.invoke({"messages": [("human", "Create a summary of our customer: Peak Performance Co.")]})
print(messages['messages'][-1].content)


**What to expect:** a natural-language summary that references the specific row(s) the tool
returned (subscription type, engagement score, renewal probability) — the agent looked the row up
itself rather than guessing.

🧪 **Try it yourself:** Ask about a different company name from the `customers` table, or add a
new row to the DataFrame and query it.

---
## Module 6 — Retrieval-Augmented Generation (RAG)

LLMs have a **knowledge cutoff** — they only know what was in their training data, and they have
no idea about your private documents. **Retrieval-Augmented Generation (RAG)** fixes this by:

1. **Loading** your documents (PDFs, text files, web pages, ...)
2. **Splitting** them into small chunks (so retrieval can be precise, and chunks fit in a prompt)
3. **Embedding** each chunk into a vector (a list of numbers capturing its meaning) and storing it
   in a **vector database**
4. At query time, **retrieving** the chunks whose embeddings are most similar to the question
5. **Generating** an answer by handing the LLM the question *plus* the retrieved chunks as context

We'll build this pipeline end to end using a real paper as our example document: *"Corrective
Retrieval Augmented Generation"* (CRAG) — conveniently, a paper about RAG itself.

In [ ]:
!pip install pypdf


In [ ]:
# Import library
from langchain_community.document_loaders import PyPDFLoader


### Document Loader

`PyPDFLoader` reads a PDF file and turns it into a list of LangChain `Document` objects (one per
page, roughly). It needs a **file path** to read from.

Below we download the CRAG paper straight from arXiv into the Colab file system — this keeps the
notebook fully self-contained, so you don't need anything already sitting in your Google Drive.
(If you'd rather use your own PDF, mount your Drive with `drive.mount('/content/drive')` and point
`PDF_PATH` at your file instead.)

In [ ]:
# 🔧 Configure: swap this for the path to your own PDF if you'd like to use different content
!wget -q -O crag_paper.pdf https://arxiv.org/pdf/2401.15884
PDF_PATH = "crag_paper.pdf"


In [ ]:
# Create a document loader for the CRAG paper
loader = PyPDFLoader(PDF_PATH)

# Load the document
data = loader.load()
print(f"Loaded {len(data)} pages")


### Splitting

Documents are usually too long to hand to an LLM (or even to embed usefully) in one piece, so we
split them into smaller **chunks**. LangChain offers several splitting strategies — let's compare
two of them on a short example before applying one to the whole paper.

In [ ]:
# Import the character splitter
from langchain_text_splitters import CharacterTextSplitter
import os


In [ ]:
quote = 'Words are flowing out like endless rain into a paper cup,\nthey slither while they pass,\nthey slip away across the universe.'


#### `CharacterTextSplitter`

The simplest splitter: it cuts the text at a single `separator` (here, a newline `\n`), and merges
the resulting pieces back together up to `chunk_size` characters, with `chunk_overlap` characters
repeated between consecutive chunks (overlap helps preserve context that would otherwise be cut in
half).

In [ ]:
chunk_size = 24
chunk_overlap = 10

# Create an instance of the splitter class
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap)

# Split the string and print the chunks
docs = splitter.split_text(quote)
print(docs)
print([len(doc) for doc in docs])


**What to expect:** because `CharacterTextSplitter` only splits on `\n`, chunks can end up longer
than `chunk_size` whenever a single line between newlines is itself longer than that — it never
splits mid-line.

#### `RecursiveCharacterTextSplitter`

A smarter, more commonly used splitter. It tries a *list* of separators in order (paragraph →
sentence → word → character), falling back to a finer-grained separator only when a chunk is still
too big — this keeps chunks closer to `chunk_size` while still respecting natural text boundaries
wherever possible.

In [ ]:
# Import the recursive character splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

quote = 'Words are flowing out like endless rain into a paper cup,\nthey slither while they pass,\nthey slip away across the universe.'
chunk_size = 24
chunk_overlap = 10

# Create an instance of the splitter class
splitter = RecursiveCharacterTextSplitter(
    separators=["\n", " ", ""],
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap)

# Split the document and print the chunks
docs = splitter.split_text(quote)
print(docs)
print([len(doc) for doc in docs])


Now let's apply `RecursiveCharacterTextSplitter` to the *real* PDF we loaded earlier, using chunk sizes suited to full paragraphs rather than this short lyric example. `split_documents` (as opposed to `split_text`) works directly on the `Document` objects from the loader, preserving their metadata (like page number).

In [ ]:
# Split the document using RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50)
docs = splitter.split_documents(data)
print(f"Split the paper into {len(docs)} chunks")


### RAG Storage & Retrieval Using a Vector Database

Now we turn each chunk into an **embedding** (a vector representation of its meaning) and store it
in **Chroma**, an open-source vector database. At query time, Chroma finds the stored chunks whose
embeddings are closest to the question's embedding — that's the "retrieval" in RAG.

In [ ]:
!pip install langchain-chroma


In [ ]:
!pip install --upgrade opentelemetry-api opentelemetry-sdk


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma


`HuggingFaceEmbeddings` runs an open-source embedding model locally — `all-MiniLM-L6-v2` is a
small, fast, widely-used sentence-embedding model, well suited to running on Colab's CPU.

In [ ]:
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


`Chroma.from_documents(...)` embeds every chunk from `docs` and stores the resulting vectors.
`.as_retriever(...)` then wraps the vector store as a LangChain **retriever** — a component with a
single job: given a query, return the most relevant documents. `search_kwargs={"k": 3}` means
"return the top 3 most similar chunks".

In [ ]:
vectorstore = Chroma.from_documents(
    docs,
    embedding=embedding_function,
    persist_directory=os.getcwd()
)

# Configure the vector store as a retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


### Putting It All Together: the RAG Chain

Finally, we assemble the full RAG pipeline with LCEL:

- `{"context": retriever, "question": RunnablePassthrough()}` — runs the retriever on the incoming
  question to fetch relevant chunks (`context`), while `RunnablePassthrough()` simply forwards the
  original question through unchanged (`question`). Both land as placeholders in the prompt below.
- The prompt template instructs the model to answer **using only the provided context** — this is
  what keeps RAG answers grounded in your documents instead of the model's general knowledge.
- The result flows into `chat_model` to produce the final answer.

In [ ]:
# Add placeholders to the message string
message = """
Answer the following question using the context provided:

Context:
{context}

Question:
{question}

Answer:
"""

# Create a chat prompt template from the message string
prompt_template = ChatPromptTemplate.from_messages([("human", message)])

# Create a chain to link retriever, prompt_template, and llm
rag_chain = ({"context": retriever, "question": RunnablePassthrough()}
            | prompt_template
            | chat_model)

# Invoke the chain
response = rag_chain.invoke("Which popular LLMs were considered in the paper?")
print(response.content)


**What to expect:** an answer that references specific models named in the CRAG paper's text —
grounded in the 3 chunks the retriever pulled back, not the LLM's general training knowledge.

🧪 **Try it yourself:** Ask `rag_chain.invoke(...)` a few different questions about the paper (e.g.
`"What is a retrieval evaluator?"` or `"What datasets were used in the experiments?"`) and compare
the answers to what you'd get by asking `chat_model` the same question directly, with no retrieval
step at all.

---
## Wrap-Up & Next Steps

You've now built, end to end:
- Direct calls to open-source LLMs, both locally (Hugging Face) and hosted (Groq)
- Reusable prompt templates, chat prompts, and few-shot prompts
- Multi-step LCEL chains
- Tool-using agents, with both built-in and custom tools
- A full Retrieval-Augmented Generation pipeline over a real PDF

### Ideas to extend this notebook
- Swap `chat_model` for a different Groq-hosted open-source model and compare answer quality/speed
- Add a second custom tool to the agent in Module 5 and see whether the model chooses correctly
  between them
- Try a different embedding model or `chunk_size`/`chunk_overlap` in Module 6 and see how it
  changes which chunks get retrieved
- Combine ideas from Modules 5 and 6: build an **agent** that has RAG retrieval itself as one of
  its tools, alongside Wikipedia or a custom tool

### Useful references
- [LangChain Python docs](https://python.langchain.com/docs/introduction/)
- [LangChain Expression Language (LCEL)](https://python.langchain.com/docs/concepts/lcel/)
- [Hugging Face Hub](https://huggingface.co/models)
- [Groq Console](https://console.groq.com/)
- [Chroma docs](https://docs.trychroma.com/)